# Tugas 1A - TF-IDF Danantara

**Author:** Muhammad Razan Parisya Putra  
**Notebook:** `Tugas 1A - TF-IDF Danantara`

---

## Objective

Memahami konsep TF-IDF secara mendalam dengan menghitung TF, IDF, dan TF-IDF
secara manual (tanpa library), kemudian membandingkan hasilnya dengan implementasi scikit-learn.

**Referensi:**  
[Colab Danantara TF-IDF](https://colab.research.google.com/drive/1cem2y7Kn3KrA44F61GOsqArHejbMga38)

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
from math import log

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

import warnings
warnings.filterwarnings('ignore')

print('All libraries loaded successfully.')

## 2. Contoh Corpus

Kita mulai dengan corpus kecil agar perhitungan manual bisa diverifikasi.

In [ ]:
# Corpus sederhana: 4 dokumen pendek
corpus = [
    'investasi danantara indonesia meningkat pesat',
    'danantara kelola dana investasi negara',
    'indonesia dorong investasi asing masuk',
    'dana kelola negara untuk rakyat indonesia',
]

print(f'Jumlah dokumen: {len(corpus)}')
for i, doc in enumerate(corpus):
    print(f'  D{i}: "{doc}"')

## 3. Perhitungan Manual TF

Term Frequency (TF) = jumlah kemunculan kata di dokumen / total kata di dokumen.

TF mengukur seberapa sering sebuah kata muncul di satu dokumen.

In [ ]:
def compute_tf(document):
    """Hitung TF untuk setiap kata di dokumen."""
    words = document.split()
    total_words = len(words)
    tf_dict = {}
    for word in words:
        tf_dict[word] = tf_dict.get(word, 0) + 1
    # Normalisasi: bagi dengan total kata
    for word in tf_dict:
        tf_dict[word] = tf_dict[word] / total_words
    return tf_dict

# Hitung TF untuk setiap dokumen
tf_results = []
for i, doc in enumerate(corpus):
    tf = compute_tf(doc)
    tf_results.append(tf)
    print(f'TF Dokumen {i}:')
    for word, score in sorted(tf.items()):
        print(f'  {word:20s} = {score:.4f}')
    print()

## 4. Perhitungan Manual IDF

Inverse Document Frequency (IDF) = log(total dokumen / jumlah dokumen yang mengandung kata).

IDF mengukur seberapa langka sebuah kata di seluruh corpus.
Kata yang muncul di semua dokumen mendapat IDF rendah (tidak penting),
kata yang jarang mendapat IDF tinggi (informatif).

In [ ]:
def compute_idf(corpus):
    """Hitung IDF untuk setiap kata unik di corpus."""
    N = len(corpus)
    idf_dict = {}
    
    # Kumpulkan semua kata unik
    all_words = set()
    for doc in corpus:
        all_words.update(doc.split())
    
    # Hitung di berapa dokumen setiap kata muncul
    for word in all_words:
        doc_count = sum(1 for doc in corpus if word in doc.split())
        idf_dict[word] = log(N / doc_count)
    
    return idf_dict

idf_results = compute_idf(corpus)

print(f'IDF (total dokumen = {len(corpus)}):')
print(f'{"Kata":20s} {"Doc Freq":>10} {"IDF":>10}')
print('-' * 42)
for word in sorted(idf_results.keys()):
    doc_count = sum(1 for doc in corpus if word in doc.split())
    print(f'{word:20s} {doc_count:>10} {idf_results[word]:>10.4f}')

## 5. Perhitungan Manual TF-IDF

TF-IDF = TF x IDF.

Kata yang sering muncul di satu dokumen (TF tinggi) tapi jarang muncul di dokumen lain
(IDF tinggi) akan mendapat skor TF-IDF tertinggi.

In [ ]:
def compute_tfidf(tf_dict, idf_dict):
    """Hitung TF-IDF = TF x IDF."""
    tfidf = {}
    for word, tf_val in tf_dict.items():
        tfidf[word] = tf_val * idf_dict.get(word, 0)
    return tfidf

# Hitung TF-IDF untuk setiap dokumen
tfidf_results = []
for i, tf in enumerate(tf_results):
    tfidf = compute_tfidf(tf, idf_results)
    tfidf_results.append(tfidf)
    print(f'TF-IDF Dokumen {i}: "{corpus[i]}"')
    for word, score in sorted(tfidf.items(), key=lambda x: x[1], reverse=True):
        print(f'  {word:20s} = {score:.4f}')
    print()

In [ ]:
# Tampilkan sebagai tabel lengkap
all_words = sorted(set(w for doc in corpus for w in doc.split()))

data = {}
for word in all_words:
    row = []
    for i in range(len(corpus)):
        row.append(round(tfidf_results[i].get(word, 0), 4))
    data[word] = row

df_tfidf_manual = pd.DataFrame(data, index=[f'D{i}' for i in range(len(corpus))]).T
print('Tabel TF-IDF Manual (baris = kata, kolom = dokumen):')
df_tfidf_manual

## 6. Implementasi dengan Scikit-Learn

Bandingkan hasil manual dengan `TfidfVectorizer` dari scikit-learn.

Catatan: scikit-learn menggunakan formula IDF yang sedikit berbeda
(smooth IDF + L2 normalization), jadi angkanya tidak akan persis sama,
tapi urutan/ranking kata terpenting akan konsisten.

In [ ]:
# Scikit-learn TF-IDF
tfidf_vec = TfidfVectorizer()
tfidf_sklearn = tfidf_vec.fit_transform(corpus)

vocab_sk = tfidf_vec.get_feature_names_out()

df_tfidf_sklearn = pd.DataFrame(
    tfidf_sklearn.toarray().round(4),
    columns=vocab_sk,
    index=[f'D{i}' for i in range(len(corpus))]
).T

print('Tabel TF-IDF Scikit-Learn (baris = kata, kolom = dokumen):')
df_tfidf_sklearn

In [ ]:
# Perbandingan: kata terpenting per dokumen (manual vs sklearn)
print('Perbandingan kata terpenting per dokumen:')
print('=' * 60)
for i in range(len(corpus)):
    # Manual
    manual_top = sorted(tfidf_results[i].items(), key=lambda x: x[1], reverse=True)[:3]
    manual_words = [w for w, s in manual_top]
    
    # Sklearn
    sk_row = tfidf_sklearn[i].toarray().flatten()
    sk_top_idx = sk_row.argsort()[::-1][:3]
    sk_words = [vocab_sk[idx] for idx in sk_top_idx]
    
    print(f'D{i}: "{corpus[i]}"')
    print(f'  Manual  : {manual_words}')
    print(f'  Sklearn : {sk_words}')
    print()

## 7. Kesimpulan

- Perhitungan manual TF-IDF dan scikit-learn menghasilkan **ranking kata terpenting yang konsisten**, meskipun angka absolutnya berbeda karena scikit-learn menggunakan smooth IDF dan L2 normalization.
- Kata seperti "investasi" yang muncul di 3 dari 4 dokumen mendapat IDF rendah (kurang informatif).
- Kata yang hanya muncul di 1 dokumen (seperti "pesat", "asing", "rakyat") mendapat IDF tinggi, menandakan kata tersebut khas untuk dokumen itu.
- TF-IDF secara efektif mengidentifikasi kata-kata yang paling merepresentasikan isi unik setiap dokumen.